# Malaysian LRRK2 analysis - Gene annotation

- **Project:** LRRK2 mutation spectrum and association study in a multi-ethnic cohort of Malaysian Parkinson’s Disease patients
- **Version:** Python/3.10.12
- **Created:** 05-NOVEMBER-2025
- **Last Update:** 12-DECEMBER-2025

## Description
1. Install required database in annovar
2. Annotation using annovar

## Getting started

### Load python libraries

In [1]:
# Import necessary packages
import os
import pandas as pd
import numpy as np
from io import StringIO
from firecloud import api as fapi
from IPython.core.display import display, HTML
import urllib.parse
from google.cloud import bigquery
import sys as sys

# Define function
# Utility routine for printing a shell command before executing it
def shell_do(command):
    print(f'Executing: {command}', file=sys.stderr)
    !$command
    
def shell_return(command):
    print(f'Executing: {command}', file=sys.stderr)
    output = !$command
    return '\n'.join(output)

/tmp/ipykernel_3557/1504764740.py:21: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


### Install Annovar

In [18]:
%%capture
%%bash

# Install ANNOVAR: We are adding the download link after registration on the annovar website
# https://www.openbioinformatics.org/annovar/annovar_download_form.php

if test -e /home/jupyter/annovar; then

echo "annovar is already installed in /home/jupyter/notebooks"
else
echo "annovar is not installed"
cd /home/jupyter/

wget http://www.openbioinformatics.org/annovar/download/0wgxR2rIVP/annovar.latest.tar.gz

tar xvfz annovar.latest.tar.gz

fi

In [19]:
%%bash 

cd /home/jupyter/annovar/

perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar refGene humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar clinvar_20250721 humandb/
perl annotate_variation.pl -buildver hg38 -downdb cytoBand humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar exac03 humandb/ 
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar avsnp151 humandb/ 
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar dbnsfp47a humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad41_genome humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar GTEx_v8_eQTL humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar GTEx_v8_sQTL humandb/

## LRRK2 annotation in Malaysian samples

### Convert PLINK format to VCF

In [34]:
%%bash 
WORK_DIR='/home/jupyter/LRRK2/release11/UMKL'
cd $WORK_DIR

label="MALAY"

# Convert binary files into vcf file
/home/jupyter/plink1.9 \
--bfile ${label}/GP2_merge_${label}_qced_updated_rm_LRRK2_nodup \
--recode vcf \
--out ${label}/GP2_merge_${label}_qced_updated_rm_LRRK2_nodup

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2_nodup.log.
Options in effect:
  --bfile MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2_nodup
  --out MALAY/GP2_merge_MALAY_qced_updated_rm_LRRK2_nodup
  --recode vcf

30088 MB RAM detected; reserving 15044 MB for main workspace.
292 variants loaded from .bim file.
926 people (544 males, 382 females) loaded from .fam.
926 phenotype values loaded from .fam.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 926 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyping rate is 0.99892.
292 variants and 926 people pass filters and QC.
Amon

In [35]:
%%bash 
WORK_DIR='/home/jupyter/LRRK2/release11/UMKL'
cd $WORK_DIR

label="MALAY"

### Bgzip and Tabix
bgzip -f ${label}/GP2_merge_${label}_qced_updated_rm_LRRK2_nodup.vcf
tabix -f -p vcf ${label}/GP2_merge_${label}_qced_updated_rm_LRRK2_nodup.vcf.gz

### Annotate VCF file


In [ ]:
%%bash 
WORK_DIR='/home/jupyter/LRRK2/release11/UMKL'
cd $WORK_DIR

label="MALAY"

perl /home/jupyter/annovar/table_annovar.pl ${label}/GP2_merge_${label}_qced_updated_rm_LRRK2_nodup.vcf.gz /home/jupyter/annovar/humandb/ \
-buildver hg38 \
-out ${label}/GP2_merge_${label}_qced_updated_rm_LRRK2_nodup.annovar \
-remove \
-protocol refGeneWithVer,avsnp151,dbnsfp47a,gnomad_genome,gnomad_exome,regeneron,allofus,clinvar_20240917,GTEx_v8_eQTL,GTEx_v8_sQTL \
-operation g,f,f,f,f,f,f,f,f,f \
-nopolish \
-nastring . \
-vcfinput

In [ ]:
shell_do(f"gsutil cp {WORK_DIR}/{label}/GP2_merge_{label}_qced_updated_rm_LRRK2_nodup.annovar.hg38_multianno.txt gs://fc-e8a73e41-545c-42b1-8720-970cf953ba35/LRRK2/release11/CHINESE/assoc/")

In [38]:
LRRK2 = pd.read_csv(f"{WORK_DIR}/{label}/GP2_merge_{label}_qced_updated_rm_LRRK2_nodup.annovar.hg38_multianno.txt",sep="\t")
LRRK2_red = LRRK2[LRRK2["Otherinfo1"] > 0]
print(LRRK2_red["ExonicFunc.refGeneWithVer"].value_counts())
print(" ")
print(LRRK2_red["Func.refGeneWithVer"].value_counts())

ExonicFunc.refGeneWithVer
.                    128
nonsynonymous SNV     22
synonymous SNV         9
Name: count, dtype: int64
 
Func.refGeneWithVer
intronic      106
exonic         31
intergenic     21
UTR3            1
Name: count, dtype: int64
